# June 2004: temperature dependence of electron transport

Reference: Tania June, John R. Evans, Graham D. Farquhar
"A simple new equation for the reversible temperature dependence of photosynthetic electron transport: a study on soybean leaf"
Functional Plant Biology 31 (2004): 275-283.

[doi: 10.1071/FP03250](https://www.doi.org/10.1071/FP03250)

The model is evaluated with the plain numpy implementation below, independent
of green-sloth's `model.ts`. Every curve is Eqn 2 with the fitted parameters of
the paper's Tables 1 and 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def june(T_leaf, J_To, To, Omega):
    """Eqn 2: J at leaf temperature T_leaf (degC)."""
    return J_To * np.exp(-(((T_leaf - To) / Omega) ** 2))


def curve(T_leaf, J_To, To, Omega, I=1200.0, a=0.9, f=0.15) -> pd.DataFrame:
    T_leaf = np.asarray(T_leaf, dtype=float)
    J = june(T_leaf, J_To, To, Omega)
    return pd.DataFrame(
        {
            "J": J,
            # Eqn 1 solved for phi_PSII
            "phi_PSII": 2 * J / (I * a * (1 - f)),
            "J_rel25": J / june(25.0, J_To, To, Omega),
        },
        index=pd.Index(T_leaf, name="T_leaf"),
    )


def family(T_leaf, curves: dict[str, tuple[float, float, float]]) -> dict[str, pd.DataFrame]:
    return {k: curve(T_leaf, *p) for k, p in curves.items()}


def plot_family(res, labels, title, xlim, ylim, ylabel=r"$J$ ($\mu$mol e$^-$ m$^{-2}$ s$^{-1}$)", col="J"):
    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    for (k, label), dash in zip(labels.items(), ["-", "--", ":", "-."]):
        ax.plot(res[k].index, res[k][col], color="black", linestyle=dash, label=label)
    ax.set(title=title, xlabel="Leaf temperature (°C)", ylabel=ylabel, xlim=xlim, ylim=ylim)
    ax.legend(frameon=False)
    fig.tight_layout()
    return fig, ax

## Figure 1: one soybean leaf

Grown at 700 µmol mol⁻¹ CO₂, 20/15 °C and 16 mM nitrate: J(To) = 257, To = 33 °C,
Ω = 23.8 °C. φPSII follows from Eqn 1 with I = 1200, a = 0.9 and f = 0.15.

In [ ]:
fig1 = curve(np.linspace(15, 45, 301), 257.0, 33.0, 23.8)
fig, ax = plt.subplots(figsize=(4.6, 4.0))
ax.plot(fig1.index, fig1["phi_PSII"], color="black")
ax.set(xlabel=r"Leaf temperature ($T_L$, °C)", ylabel=r"$\phi_{PSII}$", xlim=(15, 45), ylim=(0.3, 0.6))
twin = ax.twinx()
twin.set(ylabel=r"$J$ ($\mu$mol e$^-$ m$^{-2}$ s$^{-1}$)", ylim=(150, 275))
fig.tight_layout()
fig.savefig("figs/fig1.svg")
plt.show()

## Figure 2: nutrient treatments

(A) Soybean grown at 20/15 °C with 16, 5 and 2 mM nitrate (Table 1); (B) *Pinus radiata* with
33 and 100 % nutrient (Table 2, Walcroft et al. 1997).

In [ ]:
fig2a = family(np.linspace(15, 45, 301), {"16": (240.0, 35.4, 17.6), "5": (135.0, 36.1, 18.3), "2": (113.0, 37.7, 18.7)})
fig2b = family(np.linspace(5, 35, 301), {"LN": (47.9, 33.8, 23.8), "HN": (63.1, 34.6, 23.9)})
fig, _ = plot_family(fig2a, {"16": "16 mM", "5": "5 mM", "2": "2 mM"}, "Glycine max", (15, 45), (0, 275))
fig.savefig("figs/fig2a.svg")
fig, _ = plot_family(fig2b, {"HN": "100 % nutrient", "LN": "33 % nutrient"}, "Pinus radiata", (5, 35), (0, 70))
fig.savefig("figs/fig2b.svg")
plt.show()

## Figure 3: growth temperature

(A) Soybean grown at 20/15, 25/20 and 32/27 °C with 16 mM nitrate (Table 1); (B) *Abutilon
theophrasti* grown at 16, 20, 28 and 32 °C (Table 2, Ziska 2001).

In [ ]:
fig3a = family(np.linspace(15, 45, 301), {"20": (240.0, 35.4, 17.6), "25": (219.0, 37.8, 16.7), "32": (183.0, 39.3, 19.8)})
fig3b = family(np.linspace(10, 40, 301), {"16": (202.0, 28.4, 22.3), "20": (244.7, 29.7, 21.2), "28": (268.8, 35.2, 21.9), "32": (294.1, 37.7, 21.4)})
fig, _ = plot_family(fig3a, {"20": "20/15", "25": "25/20", "32": "32/27"}, "Glycine max", (15, 45), (0, 300))
fig.savefig("figs/fig3a.svg")
fig, _ = plot_family(fig3b, {"16": "16", "20": "20", "28": "28", "32": "32"}, "Abutilon theophrasti", (10, 40), (0, 300))
fig.savefig("figs/fig3b.svg")
plt.show()

## Figure 4: Ω against To

The fitted values of Tables 1 and 2 and the paper's regression Ω = 11.6 + 0.18 To. The
points the paper adds from Wohlfahrt et al. (1999) are not tabulated and are left out.
Hollow: tobacco grown at 25/20 and 35/28 °C, whose To lies beyond the measured range.

In [ ]:
GLYCINE = [(35.4, 17.6), (36.1, 18.3), (37.7, 18.7), (32.9, 24.1), (34.4, 22.7), (36.4, 19.4), (37.8, 16.7), (38.1, 16.2), (39.3, 19.8), (39.6, 17.1), (39.1, 18.0), (38.9, 18.2), (37.6, 14.5), (39.8, 15.7), (35.2, 16.7)]
HERBS = [(28.4, 22.3), (29.7, 21.2), (35.2, 21.9), (37.7, 21.4), (33.6, 20.5), (36.5, 17.0), (36.6, 13.9), (32.7, 25.1), (34.0, 20.7), (37.3, 18.7)]
HERBS_EXTRAPOLATED = [(55.6, 30.2), (55.7, 34.7)]
TREES = [(30.6, 16.9), (35.9, 21.6), (31.8, 22.0), (34.7, 18.7), (32.1, 16.7), (35.7, 18.5), (33.8, 23.8), (34.6, 23.9), (21.9, 14.3), (23.8, 15.0), (20.7, 13.9), (21.9, 14.3), (35.6, 12.5), (40.0, 23.6), (39.4, 22.5), (40.6, 25.8), (39.2, 16.9)]
To = np.linspace(15, 60, 451)
fig4 = pd.DataFrame({"Omega_regression": 11.6 + 0.18 * To}, index=pd.Index(To, name="To"))
fig, ax = plt.subplots(figsize=(4.6, 4.0))
ax.plot(fig4.index, fig4["Omega_regression"], color="black")
for pts, marker, face in [(GLYCINE, "^", "black"), (TREES, "o", "black"), (HERBS, "s", "black"), (HERBS_EXTRAPOLATED, "s", "none")]:
    x, y = zip(*pts)
    ax.plot(x, y, marker, color="black", markerfacecolor=face, linestyle="none")
ax.set(xlabel=r"$T_o$ (°C)", ylabel=r"$\Omega$", xlim=(15, 60), ylim=(0, 40))
fig.tight_layout()
fig.savefig("figs/fig4.svg")
plt.show()

## Figure 5: normalised responses

Eqn 2 normalised to 1 at 25 °C for (To, Ω) = (20, 15), (35, 15), (30, 15) and (30, 20).

In [ ]:
fig5 = {k: curve(np.linspace(0, 50, 501), 1.0, To_, Om) for k, (To_, Om) in {"20_15": (20, 15), "35_15": (35, 15), "30_15": (30, 15), "30_20": (30, 20)}.items()}
fig, ax = plt.subplots(figsize=(4.6, 4.0))
for k, res in fig5.items():
    ax.plot(res.index, res["J_rel25"], color="black", linestyle="--" if k == "30_20" else "-", label=k.replace("_", ", "))
ax.set(xlabel="Leaf temperature (°C)", ylabel="Relative rate", xlim=(0, 50), ylim=(0, 1.5))
ax.legend(frameon=False, title=r"$T_o$, $\Omega$")
fig.tight_layout()
fig.savefig("figs/fig5.svg")
plt.show()